In [1]:
!pip install -q transformers tokenizers

In [2]:
from tqdm import tqdm
import pandas as pd
import numpy as np
import json
import os

from collections import defaultdict

import torch
import transformers
import re

from transformers import (AutoTokenizer,
                          AutoModelForCausalLM,
                          AutoModelForMultipleChoice,
                          pipeline)

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [21]:
model_name = "gpt2-xl"
data_dir = "/content/drive/MyDrive/Master's/Second Year Grad/NLU/NLU_FinalProject/Data/JSONL_Formatted/"

data_path = "RACE-H/RACE-H_test.jsonl"
data_name = 'RACE-H_test'
save_dir = './'

Load Model & Tokenizer

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             device_map='auto')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
# check for model-specific chat template (implemented dudring pre-training)
print(tokenizer.chat_template)

None


Load and Process Data

In [22]:
df = pd.read_json(data_dir + data_path, lines=True)
df.head()

,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,high17205.txt,It is the goal of politicians everywhere-----h...,"According to the passage, we know that _ .",D,people with good facial features must be trust...,people with bad facial features could not be t...,we should judge people by their facial features,facial features might give people some wrong i...
1,high17205.txt,It is the goal of politicians everywhere-----h...,"According to Ms Cornwell, we can infer that ...",C,the science will give politicians great help,politicians could be successful with the help ...,politicians won't think highly of the science,politicians will be satisfied with the science
2,high17205.txt,It is the goal of politicians everywhere-----h...,What's the best title for the passage?,A,How Science could Help Politicians,How to Win the Trust of Voters,The Other Sides of Politicians,An Important Discovery for Politicians
3,high17226.txt,"In the 1960s, people asked about your astrolog...",The main purpose of the passage is to tell you...,B,what a website is like,how to build your own website,how to meet people online,what a website is made up of
4,high17226.txt,"In the 1960s, people asked about your astrolog...","According to the writer, your website is a pla...",D,where you can meet people all around the world,where you can buy what you want,where you can get free services,where you can meet people on the Internet


In [24]:
def create_question(row):
  input = f'''{row.prompt}

{row.question}

Please select the letter of the best answer.

A. {row.mc_a}
B. {row.mc_b}
C. {row.mc_c}
D. {row.mc_d}
'''
  return input

model_input = df.apply(create_question, axis=1)
print(model_input.iloc[0])
print(f'Number of Questions = {len(model_input):,d}')

It is the goal of politicians everywhere-----how to win and keep the trust of voters.
  Now researchers at the University of St Anurew's in Scotland say they may have the answer. They believe politicians could learn a lot from recent advances in science. A growing number of studies have shown that people do judge a book by its cover. Researchers say most of us make quick judgments about a person on the basis of how they look.
  Studies suggest that people are less likely to trust those with particularly masculine  features, such as a square jaw, small eyes or a big nose. " They are considered dominant  and less trustworthy," says Ms Cornwell. "It doesn't mean that men who look more masculine are less trustworthy-----it's just our first impression." Those with less masculine features-----larger eyes, a smaller nose and thinner lips are thought to be more trustworthy.
  The researchers are putting their science to the test at the Royal Society's annual summer exhibition in London. They h

In [25]:
answer_ids = [tokenizer(x)['input_ids'][0] for x in 'ABCD']
answer_ids

[32, 33, 34, 35]

Run Inference

In [26]:
# test output
example_question = model_input.iloc[0]

encoded_input = tokenizer(example_question, return_tensors='pt', max_length=1024, truncation=True).to(device)
output = model.generate(**encoded_input, pad_token_id=tokenizer.eos_token_id,
                        max_new_tokens=1, return_dict_in_generate=True, output_logits=True)

tokenizer.decode(output.sequences[0][encoded_input['input_ids'].shape[-1]:])

'E'

In [27]:
# get class probabilities
probs = torch.softmax(output.logits[0][:,answer_ids], dim=-1).to('cpu')
print(probs)
'ABCD'[torch.argmax(probs, dim=-1).item()]

tensor([[0.4916, 0.1133, 0.1193, 0.2757]])


'A'

In [28]:
res = defaultdict(list)

for text in tqdm(model_input):
  try:
    encoded_input = tokenizer(text, return_tensors='pt', max_length=1024, truncation=True).to(device)
    output = model.generate(**encoded_input, pad_token_id=tokenizer.eos_token_id,
                           max_new_tokens=1, return_dict_in_generate=True, output_logits=True)

    response = tokenizer.decode(output.sequences[0][encoded_input['input_ids'].shape[-1]:])
    probs = torch.softmax(output.logits[0][:,answer_ids], dim=-1)[0].to('cpu')
  except:
    response = None
    probs = None

  res['prob_A'].append(probs[0].item() if probs is not None else None)
  res['prob_B'].append(probs[1].item() if probs is not None else None)
  res['prob_C'].append(probs[2].item() if probs is not None else None)
  res['prob_D'].append(probs[3].item() if probs is not None else None)
  res['pred'].append('ABCD'[torch.argmax(probs, dim=-1).item()] if probs is not None else None)

  res['response'].append(response)

100%|██████████| 3498/3498 [27:25<00:00,  2.13it/s]


In [29]:
res_df = pd.concat([pd.DataFrame(res), df], axis=1)
res_df.head()

,prob_A,prob_B,prob_C,prob_D,pred,response,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,0.491645,0.113300,0.119322,0.275733,A,E,high17205.txt,It is the goal of politicians everywhere-----h...,"According to the passage, we know that _ .",D,people with good facial features must be trust...,people with bad facial features could not be t...,we should judge people by their facial features,facial features might give people some wrong i...
1,0.465566,0.073908,0.153238,0.307288,A,E,high17205.txt,It is the goal of politicians everywhere-----h...,"According to Ms Cornwell, we can infer that ...",C,the science will give politicians great help,politicians could be successful with the help ...,politicians won't think highly of the science,politicians will be satisfied with the science
2,0.605844,0.098547,0.087170,0.208438,A,E,high17205.txt,It is the goal of politicians everywhere-----h...,What's the best title for the passage?,A,How Science could Help Politicians,How to Win the Trust of Voters,The Other Sides of Politicians,An Important Discovery for Politicians
3,0.138777,0.076625,0.033621,0.750977,D,E,high17226.txt,"In the 1960s, people asked about your astrolog...",The main purpose of the passage is to tell you...,B,what a website is like,how to build your own website,how to meet people online,what a website is made up of
4,0.240933,0.068567,0.040858,0.649641,D,\n,high17226.txt,"In the 1960s, people asked about your astrolog...","According to the writer, your website is a pla...",D,where you can meet people all around the world,where you can buy what you want,where you can get free services,where you can meet people on the Internet


In [32]:
print(res_df.response.value_counts())

response
\n           2911
E             574
.               2
 was            2
 Late           1
*               1
You             1
 a              1
,               1
 his            1
 give           1
 found          1
 arrested       1
Name: count, dtype: int64


In [33]:
print(res_df.pred.value_counts())

pred
D    2467
A    1010
C      13
B       8
Name: count, dtype: int64


In [34]:
# evaluation metrics

# accuracy
accuracy = sum(res_df.pred == res_df.answer)/len(res_df.dropna())

# percent response failure
res_fail = sum(res_df.response.isnull())/len(res_df)

# percent response/pred match
res_pred_match = sum(res_df.dropna().pred == res_df.dropna().response)/len(res_df.dropna())

df_eval = pd.DataFrame({'dataset':[data_name],
                        'accuracy':[accuracy],
                        'response_failure':[res_fail],
                        'response_pred_match':[res_pred_match]})

if os.path.exists(f"{save_dir}benchmark_summary_{model_name}.csv"):
  df_temp = pd.read_csv(f"{save_dir}benchmark_summary_{model_name}.csv")
  df_eval = pd.concat([df_eval, df_temp], axis=0, ignore_index=True)

df_eval

,dataset,accuracy,response_failure,response_pred_match
0,RACE-H_test,0.230989,0.0,0.0
1,SAT-ACT_test,0.246212,0.0,0.0


Store Results

In [35]:
res_df.to_csv(f"{save_dir}{data_name}_benchmark_{model_name}.csv", index=False)

In [36]:
df_eval.to_csv(f"{save_dir}benchmark_summary_{model_name}.csv", index=False)